# Prompt Engineering Fundamentals - A Hands-On Introduction

This notebook introduces the core ideas behind **prompt engineering** with LangChain and OpenAI models.

You will learn:
1. How to set up an LLM in Python
2. **Zero-shot prompting** - asking the model a question directly
3. **Structured prompt templates** - breaking a prompt into Instruction, Context, Input and Output Format
4. **Few-shot prompting** - teaching the model by example
5. **Temperature** - controlling creativity and determinism
6. How to build a reusable LLM helper module

Run the cells from top to bottom. Each section explains what is happening and why.


## Section 1: Environment Setup & LLM Initialization

Before we can talk to a language model, we need three things:
- `os` - to read environment variables
- `load_dotenv` - to load API keys from a `.env` file
- `ChatOpenAI` - the LangChain wrapper around OpenAI chat models

Storing keys in a `.env` file keeps secrets out of your source code.


**Import the required libraries.**

`os` lets us access environment variables, `dotenv` loads the `.env` file, and `langchain_openai.ChatOpenAI` creates a chat-model client.


In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


**Load environment variables.**

`load_dotenv(override=True)` reads the `.env` file in the current directory and sets the corresponding environment variables. The `override=True` argument means that values from the file take precedence over any already-set variables.


In [2]:
load_dotenv(override=True)

True

**Create the chat-model client.**

`ChatOpenAI(model='gpt-5.4')` instantiates a LangChain chat model pointed at the `gpt-5.4` endpoint. The call will use the `OPENAI_API_KEY` that was loaded from `.env`.

`llm` is now reusable - we can call it as many times as we like.


In [3]:
llm = ChatOpenAI(model= "gpt-5.4")

**Test the LLM connection.**

`llm.invoke('Hi')` sends a single-turn message to the model and returns an `AIMessage` object.

The returned object contains the generated text in `.content`, plus metadata such as token usage, model name, finish reason and a unique run id. If this cell succeeds, your environment is configured correctly.


In [4]:
llm.invoke("Hi")

AIMessage(content='Hi! How can I help?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 7, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DicPaBVt1xy6a6xNDPx8b4ZCp8yrg', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e540a-d5a1-7683-b2e0-e933f0e468b8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 10, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Section 2: Zero-Shot Prompting

**Zero-shot prompting** means giving the model a task without any examples. The model must rely entirely on knowledge learned during training.

It is the simplest form of prompting and works well for questions that the model already understands.


**Ask a direct question.**

We store the question in a variable called `prompt` and pass it to `llm.invoke()`. The comment `# Zero shot prompting` reminds us that no examples are provided.


In [5]:
prompt = "What is genAI?" # Zero shot prompting

In [6]:
llm.invoke(prompt)

AIMessage(content='GenAI stands for **Generative AI**.\n\nIt refers to AI systems that can **create new content** rather than just analyze or classify existing data. That content can include:\n\n- **Text** — essays, emails, summaries, code\n- **Images** — artwork, designs, photos\n- **Audio** — music, voices, sound effects\n- **Video** — generated clips or animations\n- **Other data** — like 3D models or synthetic datasets\n\n### Simple definition\nTraditional AI often answers questions like:\n- “Is this email spam?”\n- “What object is in this image?”\n\nGenAI answers questions like:\n- “Write me a marketing email.”\n- “Create an image of a futuristic city.”\n- “Summarize this report.”\n- “Generate code for a website.”\n\n### How it works\nGenerative AI is trained on large amounts of data and learns patterns in that data. Then it uses those patterns to produce something new that resembles what it has learned.\n\n### Examples\n- **Chatbots** that write or answer questions\n- **Image gen

**Control the audience with the prompt.**

By adding *'explain it as if you are talking to a 10-year-old kid'*, we change the tone, vocabulary and depth of the answer without changing any model settings. This shows how powerful a well-chosen instruction can be.


In [ ]:
prompt = "What is genAI, explain it in such a way that you are explain to 10 year kid"
llm.invoke(prompt)

AIMessage(content='GenAI means **Generative Artificial Intelligence**.\n\nLet’s make that super simple:\n\n- **Artificial Intelligence (AI)** = a smart computer system that can do things that usually need human thinking.\n- **Generative** = something that can **create new things**.\n\nSo, **GenAI is a type of computer technology that can make stuff**, like:\n\n- stories\n- pictures\n- songs\n- videos\n- answers to questions\n- computer code\n\n### Imagine this:\nIt’s like a **super smart robot helper** that has read and seen **tons of examples** from books, pictures, websites, and more. Then, when you ask it something, it tries to **make a brand-new answer** based on what it learned.\n\n### Example:\nIf you say:\n\n> “Write me a story about a cat flying to the moon”\n\nGenAI can create a new story right away.\n\nIf you say:\n\n> “Draw a dragon eating ice cream”\n\nGenAI can make a new picture.\n\n### Is it really thinking like a person?\nNot exactly.\n\nIt does **not think or feel like

## Section 3: A Structured Prompt Template

A good prompt often has four parts:
- **Instruction** - what you want the model to do
- **Context** - background or role information
- **Input data** - the specific value the model should process
- **Output format** - how the answer should look

Structuring prompts this way makes them easier to read, debug and reuse.


**Build a reusable structured prompt.**

- `topic = input('Enter the topic')` asks the user for a topic at runtime.
- An f-string injects the topic into the `#Topic` section.
- The `#Instruction` and `#Context` sections tell the model its role and audience.
- The `#Ouput Format` section requests exactly 5 bullet points.


In [16]:
# Instruction
# Context
# Input data (NA)
# Output format

topic = input("Enter the topic")

prompt = f"""
#Instruction
You are an technical AI assistant in explaining the AI concepts

#Context
You are required to explain the concept to an advanced IT professional

#Topic
{topic}

#Ouput Format
Output to be in 5 bullet points
"""

Enter the topic genai


**Inspect the final prompt string.**

Printing `prompt` lets us see exactly what the model receives, including the user's topic. This is useful for debugging prompts before sending them to the LLM.


In [10]:
prompt

'\n#Instruction\nYou are an technical AI assistant in explaining the AI concepts\n\n#Context\nYou are required to explain the concept in such a way that a 10 year old can understand\n\n#Topic\ngenAI\n\n#Ouput Format\nOutput to be in 5 bullet points\n\n'

**Send the structured prompt to the LLM.**

`llm.invoke(prompt)` returns an `AIMessage`. We extract the generated text with `response.content` so that we can display or save it separately.


In [15]:
response = llm.invoke(prompt)
content = response.content

**Display the generated answer.**

`print(content)` renders the model's 5-bullet explanation. Because the output format was explicit, the response follows the requested structure.


In [13]:
print(content)

- **GenAI means “Generative AI”** — it is a kind of smart computer tool that can **make new things**, like stories, pictures, songs, or answers, instead of just finding old information.

- **It learns by looking at lots of examples** — like how a child learns to draw by seeing many drawings, GenAI studies **many books, images, and patterns** so it can guess what should come next.

- **It does not think like a human** — GenAI is very good at **spotting patterns** and making something that looks right, but it does not truly understand feelings or the world the way people do.

- **You give it a prompt** — a prompt is just an instruction, like “write a story about a dragon” or “draw a cat wearing glasses,” and GenAI uses that to create something new.

- **It can be helpful, but it can also make mistakes** — GenAI can help with homework, ideas, and creativity, but sometimes it gives **wrong or made-up answers**, so people should always check its work.


## Section 4: Sentiment Analysis with Zero-Shot Prompting

Now we apply zero-shot prompting to a classification task: deciding whether a movie review is positive, neutral or negative.

The prompt again uses Instruction, Context, Input and Output Format, but this time asks for a JSON result.


**Classify a positive review.**

The variable `review` holds the movie review text. The f-string places it inside the `#Input` block. The `#Ouput Format` requests JSON with a single key `sentiment`, which is easy to parse programmatically.


In [19]:
review = "This moview was quite good"
prompt = f"""
#Instruction
You are an AI assistant with the task of predicting the sentiment of the given movie review

#Context
Review should be labelled based on the sentiment

#Input
Movie Review : {review}

#Ouput Format
The output to be returned as a JSON with key as "sentiment"
"""

**Get the model's classification.**

`response.content` contains the JSON string produced by the model. For the positive review, we expect something like `'{'sentiment':'positive'}'`.


In [22]:
response = llm.invoke(prompt)
response.content

'{"sentiment":"positive"}'

**Classify a neutral review.**

Changing only the input text lets us reuse the same prompt template. The word *'okay'* is ambiguous, so this is a good test of the model's judgement.


In [23]:
review = "This movie was okay"
prompt = f"""
#Instruction
You are an AI assistant with the task of predicting the sentiment of the given movie review

#Context
Review should be labelled based on the sentiment

#Input
Movie Review : {review}

#Ouput Format
The output to be returned as a JSON with key as "sentiment"
"""
response = llm.invoke(prompt)
response.content


'{"sentiment":"neutral"}'

## Section 5: Few-Shot Prompting

**Few-shot prompting** gives the model a few examples of the desired input-output behaviour before the actual task. Examples help the model understand the exact format, style or labels you want.

This is especially useful when you need consistent output labels or a specific writing style.


**Sentiment classification with examples.**

The prompt now includes a `#Examples` block with three reviews and their labels: `Pos`, `Neutral` and `Neg`. The model learns the label format from these examples and applies it to the new review.


In [24]:
review = "This moview was quite good"
prompt = f"""
#Instruction
You are an AI assistant with the task of predicting the sentiment of the given movie review

#Context
Review should be labelled based on the sentiment

#Input
Movie Review : {review}

#Examples

Review : "I love this movie"
sentiment : "Pos"

Review : "This movie was okay"
setiment : "Neutral"

Review : "This movie was pathetic"
sentiment : "Neg"


#Ouput Format
The output to be returned as a JSON with key as "sentiment"
"""

response = llm.invoke(prompt)
response.content

'{"sentiment":"Pos"}'

**Convert a sentence to hashtags - zero-shot.**

This task asks the model to pick meaningful hashtags from a sentence and return them as JSON. Without examples, the model may include small words like *'is'* and *'the'*.


In [27]:
sentence = "AI is transforming the world"
prompt = f"""
#Instruction
You are an AI assistant with the task of converting sentences into hashtag

#Context
Convert the given sentence into hashtags, that can be suggested on the social media, keep in mind the hastags should be exact word from the sentence and
the hashtag should only have one word

#Input
Sentence : {sentence}

#Ouput Format
The output to be returned as a JSON with key as "hashtags" and the value as list of hastags
"""

response = llm.invoke(prompt)
response.content

'{"hashtags":["#AI","#is","#transforming","#the","#world"]}'

**Convert a sentence to hashtags - few-shot.**

By providing two examples, we teach the model to skip stop-words and to capitalise multi-word concepts. Compare the output with the zero-shot version above.


In [29]:
sentence = "AI is transforming the world"
prompt = f"""
#Instruction
You are an AI assistant with the task of converting sentences into hashtag

#Context
Convert the given sentence into hashtags, that can be suggested on the social media, keep in mind the hastags should be exact word from the sentence and
the hashtag should only have one word

#Input
Sentence : {sentence}

#Few shot examples
Sentence : "Machine learning is powerful
Hastags : ["#MachineLearning" ,"#Powerful"]

Sentence : "AI is draining the water resources around the world
Hastags : ["AI", "#WaterResources", "#World", "#draining"]

#Ouput Format
The output to be returned as a JSON with key as "hashtags" and the value as list of hastags
"""

response = llm.invoke(prompt)
response.content

'{"hashtags":["#AI","#Transforming","#World"]}'

## Section 6: Temperature & Creativity Control

Language models generate text one token at a time. At each step, the model assigns probabilities to possible next words. **Temperature** controls how those probabilities are used:

- `temperature = 0` - the model picks the highest-probability token every time. Output is deterministic and focused.
- `temperature = 1` - the model samples more freely. Output is more creative and varied.

Lower temperatures are good for structured tasks; higher temperatures for brainstorming or creative writing.


**Conceptual overview of next-token prediction.**

The comments in this cell illustrate the idea behind greedy decoding (`temperature=0`) versus random sampling. They are not executable code, but they explain why temperature matters.


In [30]:
# The sky is 

# Corpus - all the words (tokens) that it is trained now

# Assign the probabilities to each of this word

# Blue - 0.9
# white - 0.7
# cloudy - 0.6
# beuatiful - 0.4

# Greedy technique - "Blue"
# Random - 
# Top K - 

# Temperature - 0: Greedy technique - Deterministic
# Temperature - 1 : Random - Creative

# Temperature - 0 - 1


**Create two LLM instances with different temperatures.**

`llm1` uses `temperature=0` for deterministic responses, while `llm2` uses `temperature=1` for more varied responses. Both point to the same model, so any difference in output comes only from the temperature setting.


In [31]:
llm1 = ChatOpenAI(model = "gpt-5.4", temperature = 0)
llm2 = ChatOpenAI(model = "gpt-5.4", temperature = 1)

**Run the deterministic model.**

`llm1.invoke('The product is okay but expensive')` should give a focused, predictable response.


In [32]:
llm1.invoke("The product is okay but expensive")

AIMessage(content='Here are a few concise rewrites, depending on tone:\n\n- The product is fine, but it’s overpriced.\n- The product is decent, but too expensive.\n- The product is okay, but the price is too high.\n- The product works well enough, but it costs too much.\n\nIf you want, I can make it sound more professional, polite, or stronger for a review.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 12, 'total_tokens': 94, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-Did37VrS9LNNLFCUoaKa6mVX00m4l', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e5430-3f6a-71a2-a1a0-796ffc147710-0', tool_calls=[], invalid_tool_ca

**Run the creative model.**

`llm2.invoke(...)` with `temperature=1` may produce a longer or more diverse response. Run it multiple times to see how the output changes.


In [33]:
llm2.invoke("The product is okay but expensive")

AIMessage(content='It sounds like the product quality is acceptable, but the price feels too high for what it offers.\n\nHere are a few polished ways to say it:\n\n- **The product is decent, but it’s overpriced.**\n- **The product is okay, but it doesn’t feel worth the price.**\n- **The quality is acceptable, but the price is too high.**\n- **It’s an okay product, but it’s too expensive for what you get.**\n\nIf you want, I can also help rewrite this as:\n- a **customer review**\n- a **formal complaint**\n- a **shorter phrase**\n- a **more professional sentence**', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 135, 'prompt_tokens': 12, 'total_tokens': 147, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprin

## Section 7: Building Reusable LLM Utilities

In a real project, you do not want to repeat import statements and API-key loading in every script. Instead, create a small helper module that centralises LLM creation and invocation.

This notebook uses a local package called `llm` with two helper functions: `get_llm()` and `invoke_llm()`.


**Project-structure notes.**

The comments in this cell sketch a clean layout:
- A module for LLM wrappers and endpoints
- A module for prompt utilities

A reusable helper makes the rest of the codebase simpler and easier to maintain.


In [ ]:
# Main

## LLM
    # -- Wrappers, LLM end points for multiple providers
## Prompts
    # prompts.py



# We should build one script - we can reuse

**Import the helper functions.**

`from llm.llm import get_llm, invoke_llm` loads the reusable functions. If this import fails, make sure the `llm` package is on the Python path and contains an `llm.py` file.


In [34]:
from llm.llm import get_llm, invoke_llm

**Use the helper to generate an answer.**

- `get_llm(os.getenv('OPENAI_API_KEY'))` creates an LLM client using the API key from the environment.
- `invoke_llm('You are an AI Assistant', 'What is GenAI', llm)` sends a system-style instruction and a user question.


In [36]:
llm = get_llm(os.getenv("OPENAI_API_KEY"))
response = invoke_llm("You are an AI Assistant", "What is GenAI",llm)

**Display the helper's response.**

The helper returns the generated text directly, so we can inspect it by evaluating `response`. Notice how the reusable wrapper hides the low-level `AIMessage` object from the caller.


In [37]:
response

'GenAI, or Generative AI, refers to a class of artificial intelligence models and systems that are designed to generate new content, data, or information based on the patterns and structures learned from existing data. This can include text, images, music, and other forms of media. Generative AI uses techniques such as deep learning, neural networks, and natural language processing to create outputs that can mimic human-like creativity and expression.\n\nSome common applications of Generative AI include:\n\n1. **Text Generation**: Models like GPT-3 and GPT-4 can generate human-like text for various applications, including chatbots, content creation, and storytelling.\n\n2. **Image Generation**: Tools like DALL-E and Midjourney can create images from textual descriptions, allowing users to generate artwork or visual content based on their prompts.\n\n3. **Music Composition**: AI systems can compose original music by learning from existing compositions and styles.\n\n4. **Video Generatio

---

**End of notebook.**

You have now seen zero-shot prompting, structured prompts, few-shot examples, temperature control and a reusable LLM helper. Use these patterns as building blocks for more complex agentic applications.
